# Project 6: electricity per person in Suriname and three neighbours

**The question.** How does Suriname's electricity use per person compare with that of Guyana, Trinidad and Tobago and Jamaica? Does the ranking change when you divide each country's total by its population?

**The data.** The energy dataset of Our World in Data, which takes electricity demand from Ember's Yearly Electricity Data (2026) and population from its own population series. Our World in Data last updated the file on 2026-09-30, and the course copy was downloaded on 2026-11-06. The table keeps the four countries from 2000 onward.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week the data checker compares every figure per person with the publisher's own column.

**What you hand in on Friday.** This notebook with every CHECK passing, one chart with two panels, a note of about 200 words for a ministry official who plans Suriname's electricity supply, and your AI log.

## Load the data

This cell is given. It reads the table from the course folder, or straight from Our World in Data if the file is not there.

In [ ]:
import pandas as pd

URL = "https://catalog.ourworldindata.org/energy/owid_energy/owid_energy.csv"
COLUMNS = ['country', 'iso_code', 'year', 'population', 'electricity_demand_twh', 'electricity_demand_per_capita_kwh']
COUNTRIES = ['SUR', 'GUY', 'TTO', 'JAM']
try:
    energy = pd.read_csv("https://santerre.ai/bigi-data/data/week06-electricity.csv")
except FileNotFoundError:
    raw = pd.read_csv(URL, usecols=COLUMNS)
    energy = (raw[raw["iso_code"].isin(COUNTRIES) & (raw["year"] >= 2000)][COLUMNS]
              .sort_values(["country", "year"]).reset_index(drop=True))
energy

## TODO 1: describe the table

Print the number of rows, the countries, the first and last year, and the missing values in each column. Then store in `latest` the last year in which all four countries have a value for electricity demand. Look at the last year in the table before you decide.

In [ ]:
# TODO 1
latest = None

In [ ]:
# CHECK 1
counts = energy.dropna(subset=["electricity_demand_twh"]).groupby("year")["country"].nunique()
expected_latest = int(counts[counts == 4].index.max())
assert latest is not None, "latest is still empty"
assert int(latest) == expected_latest, "latest must be the last year in which all four countries have a demand value"
print("CHECK 1 passed:", expected_latest)

## TODO 2: rank by total use

Store in `rank_total` a list of the four countries, from the highest electricity demand in `latest` to the lowest.

In [ ]:
# TODO 2
rank_total = None

In [ ]:
# CHECK 2
in_latest = energy[energy["year"] == expected_latest].set_index("country")
total_order = list(in_latest["electricity_demand_twh"].sort_values(ascending=False).index)
assert rank_total is not None, "rank_total is still empty"
assert list(rank_total) == total_order
print("CHECK 2 passed:", total_order)

## TODO 3: kilowatt-hours per person

Add a column `kwh_per_person` that divides each row's electricity demand by the population in the same row. Demand is in terawatt-hours (TWh), and one TWh is 1,000,000,000 kilowatt-hours (kWh).

The table also holds the publisher's own figure, `electricity_demand_per_capita_kwh`. The CHECK compares yours with it.

In [ ]:
# TODO 3

In [ ]:
# CHECK 3
assert "kwh_per_person" in energy.columns, "add the column kwh_per_person first"
own = energy["electricity_demand_twh"] * 1e9 / energy["population"]
assert ((energy["kwh_per_person"] - own).abs() < 0.01).sum() == own.notna().sum(), "divide each row by the population in the same row"
gap = (energy["kwh_per_person"] - energy["electricity_demand_per_capita_kwh"]).abs().max()
assert gap < 1, "your figures differ from the publisher's: check the unit conversion"
print("CHECK 3 passed: the largest gap from the publisher's figure is", round(gap, 3), "kWh")

## TODO 4: rank per person, and the years the order was different

Store in `rank_per_person` the four countries from the highest kWh per person in `latest` to the lowest. Store in `moved` the countries whose place differs from their place in `rank_total`.

Then store in `order_years` every year in which the order per person of the four countries differs from the order in `latest`.

In [ ]:
# TODO 4
rank_per_person = None
moved = None
order_years = None

In [ ]:
# CHECK 4
pp = energy.assign(pp=energy["electricity_demand_twh"] * 1e9 / energy["population"]).dropna(subset=["pp"])
def order_in(year):
    return list(pp[pp["year"] == year].sort_values("pp", ascending=False)["country"])
assert rank_per_person is not None and moved is not None and order_years is not None, "one of the three is still empty"
assert list(rank_per_person) == order_in(expected_latest)
assert sorted(moved) == sorted(c for c in total_order if total_order.index(c) != order_in(expected_latest).index(c))
expected_years = [int(y) for y in sorted(pp["year"].unique()) if order_in(y) != order_in(expected_latest)]
assert sorted(int(y) for y in order_years) == expected_years
print("CHECK 4 passed:", order_in(expected_latest), sorted(moved), expected_years)

## TODO 5: one chart, two panels

Draw one figure with two bar charts side by side for `latest`: total demand in TWh on the left and kWh per person on the right. Keep the countries in the same order in both panels, give each panel a title and a labelled axis, and name the source on the figure.

In [ ]:
# TODO 5

## TODO 6: growth since 2000

Store in `growth_total` the percentage by which Suriname's total demand grew from 2000 to `latest`, and in `growth_per_person` the percentage for its kWh per person. Print Suriname's population growth over the same years, and explain in one sentence why the two growth figures differ.

In [ ]:
# TODO 6
growth_total = None
growth_per_person = None

In [ ]:
# CHECK 6
sur = energy[energy["country"] == "Suriname"].set_index("year")
sur_kwh = sur["electricity_demand_twh"] * 1e9 / sur["population"]
assert growth_total is not None and growth_per_person is not None, "growth_total or growth_per_person is still empty"
assert abs(growth_total - (sur.loc[expected_latest, "electricity_demand_twh"] / sur.loc[2000, "electricity_demand_twh"] - 1) * 100) < 0.1
assert abs(growth_per_person - (sur_kwh[expected_latest] / sur_kwh[2000] - 1) * 100) < 0.1
print("CHECK 6 passed:", round(growth_total, 1), "percent in total and", round(growth_per_person, 1), "percent per person")

## TODO 7: the AI check

Write a function `check_claim(country, year, claimed_kwh)` that returns how far a claimed figure in kWh per person sits from the table's figure for that country and year, as a percentage of the table's figure. A claim that is 10% too high returns 10.

Then ask your AI assistant, without showing it the table: *"Rank Suriname, Guyana, Trinidad and Tobago and Jamaica by electricity use per person. For each country, give the figure in kWh, the year it is for, and the population you divided by."* Write its answer in your AI log.

Test each figure with `check_claim` for the year the AI named. Compare each population it used with the table's population for that year, and compare its order with `rank_per_person` and `order_years`. Note whether it ranked the countries by total use when you asked about use per person.

In [ ]:
# TODO 7
check_claim = None

In [ ]:
# CHECK 7
assert check_claim is not None, "check_claim is still empty"
sur = energy[energy["country"] == "Suriname"].set_index("year")
right = sur.loc[expected_latest, "electricity_demand_twh"] * 1e9 / sur.loc[expected_latest, "population"]
assert abs(check_claim("Suriname", expected_latest, right)) < 0.01
wrong = sur.loc[expected_latest, "electricity_demand_twh"] * 1e9 / sur.loc[2000, "population"]
assert abs(check_claim("Suriname", expected_latest, wrong) - (wrong / right - 1) * 100) < 0.01
print("CHECK 7 passed: dividing by the population of 2000 makes Suriname's figure", round((wrong / right - 1) * 100, 1), "percent too high")

## TODO 8: the note for an official

In a text cell below, write about 200 words for a ministry official who plans Suriname's electricity supply. Compare Suriname with the three other countries in total and per person, use the numbers you computed, name the source, and say one thing the data cannot tell the official.